In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("/content/complete_email_analytics_dataset_10000.csv")

In [3]:
print(df.head())

  email_id                   sender                  subject  \
0   E06253  customer266@example.com  Product arrived damaged   
1   E04685  customer835@example.com  Product arrived damaged   
2   E01732  customer320@example.com           Payment failed   
3   E04743  customer968@example.com            Billing issue   
4   E04522  customer630@example.com           Payment failed   

                                                body   category sentiment  \
0  I am unhappy with my recent experience. Please...  Complaint  Negative   
1  I am unhappy with my recent experience. Please...  Complaint  Negative   
2  I am facing a technical problem and would appr...    Support   Neutral   
3  I am unhappy with my recent experience. Please...  Complaint  Negative   
4  I am facing a technical problem and would appr...    Support  Negative   

  priority           created_at  response_time_hours  
0     High  2026-02-04 03:59:00                 2.56  
1     High  2026-01-14 12:57:00           

In [4]:
print(df.shape)

(10000, 9)


In [5]:
print(df.columns)

Index(['email_id', 'sender', 'subject', 'body', 'category', 'sentiment',
       'priority', 'created_at', 'response_time_hours'],
      dtype='object')


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   email_id             10000 non-null  object 
 1   sender               10000 non-null  object 
 2   subject              10000 non-null  object 
 3   body                 10000 non-null  object 
 4   category             10000 non-null  object 
 5   sentiment            10000 non-null  object 
 6   priority             10000 non-null  object 
 7   created_at           10000 non-null  object 
 8   response_time_hours  10000 non-null  float64
dtypes: float64(1), object(8)
memory usage: 703.3+ KB


In [7]:
df.isnull().sum()

,0
email_id,0
sender,0
subject,0
body,0
category,0
sentiment,0
priority,0
created_at,0
response_time_hours,0


In [8]:
df.duplicated().sum()

np.int64(0)

text cleaning


In [9]:
df["sender"]=df["sender"].str.strip().str.lower()

In [10]:
df["subject"]=df["subject"].str.strip()

Date Conversion

In [11]:
df["created_at"]=pd.to_datetime(df["created_at"],errors="coerce")

response time validation

In [12]:
df["response_time_hours"].describe()

,response_time_hours
count,10000.000000
mean,5.752402
std,3.869389
min,0.200000
25%,2.660000
50%,5.040000
75%,7.840000
max,22.080000


EDA

In [13]:
Total_Emails=df.shape[0]
Total_Emails

10000

Category Analysis

In [14]:
df["category"].value_counts()

,count
category,
Complaint,2547
Support,2298
Sales,1730
Feedback,1193
Job,841
General,820
Spam,571


sentiment analysis


In [15]:
df["sentiment"].value_counts()

,count
sentiment,
Negative,4437
Neutral,2838
Positive,2725


priority analysis


In [16]:
df["priority"].value_counts()

,count
priority,
Medium,4071
High,3118
Low,2811


response time analysis

In [17]:
df["response_time_hours"].mean()

np.float64(5.752402)

category VS response time

In [18]:
df.groupby("category")["response_time_hours"].mean()

,response_time_hours
category,
Complaint,4.754107
Feedback,6.896857
General,6.743841
Job,6.716112
Sales,6.497445
Spam,7.018651
Support,4.682737


In [19]:
!pip install -q langchain langchain-groq langgraph pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.3 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
userdata.get('GROQ_API_KEY')

In [21]:
from langchain_groq import ChatGroq

In [22]:
llm=ChatGroq(
    model="llama-3.1-8b-instant",temperature=0,
    api_key=userdata.get('GROQ_API_KEY')
)

In [23]:
email="""
Subject: Order was delayed
My order was supposed to arrive three days  ago,
but I  still  haven't received it.
Please help.
"""

In [24]:
response=llm.invoke(
    f"""
    Analyze this customer email: {email}
    Tell me:
    1.Category
    2.Priority
    3.Sentiment
    4.Response Time in hours
    5.Suggested reply
    """
)
print(response.content)

Based on the given customer email, here's the analysis:

1. **Category**: The category of this email is **Order Issue** or **Shipping Delay**.

2. **Priority**: The priority of this email is **High**. The customer is expressing frustration and concern about not receiving their order on time, which indicates a high level of urgency.

3. **Sentiment**: The sentiment of this email is **Negative**. The customer is unhappy with the delay in their order and is seeking help, which indicates a negative emotional tone.

4. **Response Time in hours**: Assuming a standard business day (8 hours) and a 24/7 support system, the response time should be within 2-4 hours. However, this may vary depending on the company's support policies and availability.

5. **Suggested reply**: Here's a suggested reply to the customer:

"Dear [Customer],

Thank you for reaching out to us about the delay in your order. We apologize for the inconvenience this has caused and are here to help. We will look into the statu

In [25]:
from pydantic  import BaseModel,Field
class EmailAnalysis(BaseModel):
  category: str = Field(description="Email Category")
  priority: str = Field(description="High,Medium,Low")
  sentiment: str = Field(description="Positive,Neutral,or Negative")
  summary:str=Field(description="Short emails summary")
  suggested_reply:str=Field(description="Professional customer reply")

In [26]:
structured_llm=llm.with_structured_output(EmailAnalysis)

In [27]:
result=structured_llm.invoke(
    f"""
    Analyze the following customer email.
    Email:{email}

    Allowed categories:
    Complaint,Support,Sales,Feedback,Job,General,Spam

    Allowed sentiment:
    Positive,Neutral,Negative

    Allowed priority:
    High,Medium,Low
    Return:
    category,priority,sentiment,summary,suggested_reply
    """
)
print(result)

category='Complaint' priority='High' sentiment='Negative' summary='Customer is unhappy with delayed order' suggested_reply='Sorry to hear that your order was delayed. Can you please provide your order number so we can look into this further and provide a resolution?'


In [28]:
import pandas as pd
df=pd.read_csv('/content/complete_email_analytics_dataset_10000.csv')
df.head()

,email_id,sender,subject,body,category,sentiment,priority,created_at,response_time_hours
0,E06253,customer266@example.com,Product arrived damaged,I am unhappy with my recent experience. Please...,Complaint,Negative,High,2026-02-04 03:59:00,2.56
1,E04685,customer835@example.com,Product arrived damaged,I am unhappy with my recent experience. Please...,Complaint,Negative,High,2026-01-14 12:57:00,2.03
2,E01732,customer320@example.com,Payment failed,I am facing a technical problem and would appr...,Support,Neutral,High,2025-07-10 07:26:00,1.36
3,E04743,customer968@example.com,Billing issue,I am unhappy with my recent experience. Please...,Complaint,Negative,High,2025-07-22 04:52:00,1.97
4,E04522,customer630@example.com,Payment failed,I am facing a technical problem and would appr...,Support,Negative,Low,2025-06-24 03:03:00,5.35


In [29]:
def analyze_email(subject,body):
  email_text=f"""
  Subject:{subject}
  Body:{body}
  """
  result=structured_llm.invoke(
      f"""
      Analyze the following customer email.
      Email:{email_text}
      Allowed categories:
      Complaint,Support,Sales,Feedback,Job,General,Spam

      Allowed sentiment:
      Positive,Neutral,Negative

      Allowed priority:
      High,Medium,Low

      """
  )
  return result

In [33]:
result=analyze_email(df.iloc[0]["subject"],df.iloc[0]["body"])
print(result)

category='Complaint' priority='High' sentiment='Negative' summary='Customer is unhappy with their recent experience and has received a damaged product.' suggested_reply='Dear valued customer, we apologize for the inconvenience caused by the damaged product. We will review this issue and help you resolve it. Please provide us with your order reference number, ORD-50194, so we can assist you further. Thank you for your patience and cooperation.'


In [64]:
results=[]
for i in range(10):
  result=analyze_email(df.iloc[i]["subject"],df.iloc[i]["body"])
  results.append({
      "category_ai":result.category,
      "priority_ai":result.priority,
      "sentiment_ai":result.sentiment,
      "summary_ai":result.summary,
      "suggested_reply_ai":result.suggested_reply
  })


In [65]:
ai_df=pd.DataFrame(results)
ai_df

,category_ai,priority_ai,sentiment_ai,summary_ai,suggested_reply_ai
0,Complaint,High,Negative,Customer is unhappy with their recent experien...,"Dear valued customer, we apologize for the inc..."
1,Complaint,High,Negative,Customer is unhappy with their recent experien...,"Dear valued customer, we apologize for the inc..."
2,Complaint,High,Negative,Customer is facing a technical problem and nee...,Thank you for reaching out to us. Can you plea...
3,Complaint,High,Negative,Customer is unhappy with recent experience and...,"Dear valued customer, we apologize for the inc..."
4,Complaint,High,Negative,Customer is facing a technical problem and nee...,Thank you for reaching out to us. Can you plea...
5,Job,Low,Positive,Customer is applying for Data Analyst role,Thank you for your interest in the Data Analys...
6,Complaint,High,Negative,Customer is unhappy with recent experience and...,"Dear valued customer, we apologize for the poo..."
7,Complaint,High,Negative,Customer is unhappy with their recent experien...,"Dear valued customer, we apologize for the del..."
8,Complaint,High,Negative,Customer is unhappy with their recent experien...,"Dear valued customer, we apologize for the poo..."
9,Job,Low,Positive,Customer is applying for Business Analyst posi...,Thank you for your interest in the Business An...


AI Accuracy Analysis

In [66]:
comparison=pd.DataFrame({
    "actual_category":df.iloc[:10]["category"].values,
    "ai_category":ai_df["category_ai"]
})
comparison

,actual_category,ai_category
0,Complaint,Complaint
1,Complaint,Complaint
2,Support,Complaint
3,Complaint,Complaint
4,Support,Complaint
5,Job,Job
6,Complaint,Complaint
7,Complaint,Complaint
8,Complaint,Complaint
9,Job,Job


In [67]:
accuracy=(
    comparison["actual_category"]==comparison["ai_category"]
).mean()

print("Category Accuracy:",accuracy)


Category Accuracy: 0.8


In [68]:
final_df=pd.concat(
    [df.iloc[:len(ai_df)].reset_index(drop=True),
     ai_df.reset_index(drop=True)],axis=1
)
final_df.head()

,email_id,sender,subject,body,category,sentiment,priority,created_at,response_time_hours,category_ai,priority_ai,sentiment_ai,summary_ai,suggested_reply_ai
0,E06253,customer266@example.com,Product arrived damaged,I am unhappy with my recent experience. Please...,Complaint,Negative,High,2026-02-04 03:59:00,2.56,Complaint,High,Negative,Customer is unhappy with their recent experien...,"Dear valued customer, we apologize for the inc..."
1,E04685,customer835@example.com,Product arrived damaged,I am unhappy with my recent experience. Please...,Complaint,Negative,High,2026-01-14 12:57:00,2.03,Complaint,High,Negative,Customer is unhappy with their recent experien...,"Dear valued customer, we apologize for the inc..."
2,E01732,customer320@example.com,Payment failed,I am facing a technical problem and would appr...,Support,Neutral,High,2025-07-10 07:26:00,1.36,Complaint,High,Negative,Customer is facing a technical problem and nee...,Thank you for reaching out to us. Can you plea...
3,E04743,customer968@example.com,Billing issue,I am unhappy with my recent experience. Please...,Complaint,Negative,High,2025-07-22 04:52:00,1.97,Complaint,High,Negative,Customer is unhappy with recent experience and...,"Dear valued customer, we apologize for the inc..."
4,E04522,customer630@example.com,Payment failed,I am facing a technical problem and would appr...,Support,Negative,Low,2025-06-24 03:03:00,5.35,Complaint,High,Negative,Customer is facing a technical problem and nee...,Thank you for reaching out to us. Can you plea...


In [69]:
final_df["category_match"]=(final_df["category"]==final_df["category_ai"])
final_df["priority_match"]=(final_df["priority"]==final_df["priority_ai"])
final_df["sentiment_match"]=(final_df["sentiment"]==final_df["sentiment_ai"])

In [72]:
final_df[
    [
        "category","category_ai","category_match",
        "priority","priority_ai","priority_match",
        "sentiment","sentiment_ai","sentiment_match"
    ]
].head(10)

,category,category_ai,category_match,priority,priority_ai,priority_match,sentiment,sentiment_ai,sentiment_match
0,Complaint,Complaint,True,High,High,True,Negative,Negative,True
1,Complaint,Complaint,True,High,High,True,Negative,Negative,True
2,Support,Complaint,False,High,High,True,Neutral,Negative,False
3,Complaint,Complaint,True,High,High,True,Negative,Negative,True
4,Support,Complaint,False,Low,High,False,Negative,Negative,True
5,Job,Job,True,Low,Low,True,Negative,Positive,False
6,Complaint,Complaint,True,High,High,True,Negative,Negative,True
7,Complaint,Complaint,True,High,High,True,Negative,Negative,True
8,Complaint,Complaint,True,Medium,High,False,Negative,Negative,True
9,Job,Job,True,Medium,Low,False,Neutral,Positive,False


In [73]:
category_accuracy=(final_df["category_match"]).mean()
priority_accuracy=(final_df["priority_match"]).mean()
sentiment_accuracy=(final_df["sentiment_match"]).mean()

print("Category Accuracy:",round(category_accuracy*100,2),"%")
print("Priority Accuracy:",round(priority_accuracy*100,2),"%")
print("Sentiment Accuracy:",round(sentiment_accuracy*100,2),"%")

Category Accuracy: 80.0 %
Priority Accuracy: 70.0 %
Sentiment Accuracy: 70.0 %


In [74]:
final_df.to_csv("Email_Analytics_AI_Final.csv",index=False)

In [75]:
final_df.shape

(10, 17)